In [10]:
import pyodbc

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=US HEALTH CARE;"
    "Trusted_Connection=yes;"
)

cursor = conn.cursor()
cursor.execute("SELECT TOP 5 * FROM claims")
rows = cursor.fetchall()

print(rows)

[(1, 1001, 201, 'B01', '90001', datetime.date(2026, 4, 17), 498.32), (2, 1002, 202, 'C02', '90002', datetime.date(2026, 4, 16), 139.09), (3, 1003, 203, 'D03.3', '90003', datetime.date(2026, 4, 15), 68.11), (4, 1004, 204, 'E04', '90004', datetime.date(2026, 4, 14), 477.71), (5, 1005, 205, 'F05', '90005', datetime.date(2026, 4, 13), 209.78)]


In [15]:
import pyodbc

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=US HEALTH CARE;"
    "Trusted_Connection=yes;"
)

print("Connection successful")

Connection successful


In [16]:
import pandas as pd
import sqlalchemy

engine = sqlalchemy.create_engine(
    "mssql+pyodbc:///?odbc_connect="
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=US HEALTH CARE;"
    "Trusted_Connection=yes;"
)

df = pd.read_sql("SELECT TOP 10 * FROM dbo.claims", engine)

In [50]:
df

,claim_id,patient_id,provider_id,icd_code,cpt_code,service_date,billing_amount
0,1,1001.0,201.0,B01,90001,2026-04-17,498.32
1,2,1002.0,202.0,C02,90002,2026-04-16,139.09
2,3,1003.0,203.0,D03.3,90003,2026-04-15,68.11
3,4,1004.0,204.0,E04,90004,2026-04-14,477.71
4,5,1005.0,205.0,F05,90005,2026-04-13,209.78
...,...,...,...,...,...,...,...
107,108,1008.0,208.0,E18.0,None,2025-12-31,716.09
108,109,1009.0,209.0,F19,90109,2025-12-30,689.53
109,110,1010.0,210.0,12345,1234,None,91.33
110,111,1011.0,211.0,H21.3,90111,2025-12-28,243.44


In [51]:
missing = df.isnull().sum()

In [52]:
duplicates = df[df.duplicated(
    subset=['patient_id','service_date','cpt_code','provider_id'],
    keep=False
)]

In [53]:
# ICD validation
invalid_icd = df[
    ~df['icd_code'].astype(str).str.match(r'^[A-Z]\d{2}(\.\d+)?$')
]

In [54]:
# CPT validation
invalid_cpt = df[
    ~df['cpt_code'].astype(str).str.match(r'^\d{5}$')
]

In [55]:
# Billing issues
negative_billing = df[df['billing_amount'] < 0]


In [57]:
# Future dates
df['service_date'] = pd.to_datetime(df['service_date'], errors='coerce')

future_dates = df[df['service_date'] > pd.Timestamp.now()]

In [58]:
# 4. SUMMARY REPORT
# =========================
summary = pd.DataFrame([{
    "total_records": len(df),
    "missing_values": missing.sum(),
    "duplicates": len(duplicates),
    "invalid_icd": len(invalid_icd),
    "invalid_cpt": len(invalid_cpt),
    "negative_billing": len(negative_billing),
    "future_dates": len(future_dates)
}])

print("\nQA SUMMARY:")
print(summary)


QA SUMMARY:
   total_records  missing_values  duplicates  invalid_icd  invalid_cpt  \
0            112              27           0           15           17   

   negative_billing  future_dates  
0                 3             0  


In [59]:
output_file = "healthcare_qa_report.xlsx"

with pd.ExcelWriter(output_file) as writer:
    summary.to_excel(writer, sheet_name="summary", index=False)
    duplicates.to_excel(writer, sheet_name="duplicates", index=False)
    invalid_icd.to_excel(writer, sheet_name="invalid_icd", index=False)
    invalid_cpt.to_excel(writer, sheet_name="invalid_cpt", index=False)
    negative_billing.to_excel(writer, sheet_name="billing_issues", index=False)
    future_dates.to_excel(writer, sheet_name="future_dates", index=False)

print("\n✅ REPORT GENERATED SUCCESSFULLY:", output_file)


✅ REPORT GENERATED SUCCESSFULLY: healthcare_qa_report.xlsx
